# 03. Training (Text Cell)

ADR-016 §3-2 / ADR-017 §3-2 모델 5종 × polluter 5종 × level 6 × dataset 6 학습.

분류 트랙: LogReg+TFIDF / TextCNN / DistilBERT / BERT-base / RoBERTa-base
회귀 트랙: Ridge+TFIDF / XGBoost+TFIDF / TextCNN-Reg / DistilBERT-Reg / BERT-base-Reg

총 학습 건수: 6 dataset × 5 model × 5 polluter × 6 level + 6×5 baseline = 930건
T4 기준 transformer ≈ 5~20분 → 30~50시간 (분류) + 30~50시간 (회귀)

Output: `results/text_train_metrics.csv` (rows = dataset × model × polluter × level × seed)

---


## 0. import + 모델 정의


In [ ]:
import sys, os
ROOT = '/content/drive/MyDrive/capstone/dsc'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, r2_score


## 1. 모델 학습 함수

각 모델은 (train_texts, train_labels, test_texts, test_labels) → metric 반환.


In [ ]:
def train_logreg_tfidf(tr_t, tr_y, te_t, te_y, max_features=20000):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(tr_t); Xte = vec.transform(te_t)
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, random_state=42).fit(Xtr, tr_y)
    return accuracy_score(te_y, clf.predict(Xte))


def train_ridge_tfidf(tr_t, tr_y, te_t, te_y, max_features=20000, alpha=1.0):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(tr_t); Xte = vec.transform(te_t)
    clf = Ridge(alpha=alpha, random_state=42).fit(Xtr, tr_y)
    return r2_score(te_y, clf.predict(Xte))


def train_xgb_tfidf(tr_t, tr_y, te_t, te_y, max_features=20000):
    from xgboost import XGBRegressor
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    Xtr = vec.fit_transform(tr_t); Xte = vec.transform(te_t)
    clf = XGBRegressor(max_depth=6, n_estimators=500, learning_rate=0.05,
                        random_state=42, n_jobs=-1).fit(Xtr, tr_y)
    return r2_score(te_y, clf.predict(Xte))


In [ ]:
# TextCNN — 분류/회귀 head 교체
class TextCNN(nn.Module):
    def __init__(self, vocab_size, n_class, emb=128, kernels=(3, 4, 5), filters=100,
                 dropout=0.5, regression=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb, filters, k, padding=k // 2) for k in kernels])
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(filters * len(kernels), 1 if regression else n_class)
        self.regression = regression

    def forward(self, x):
        x = self.emb(x).transpose(1, 2)
        x = torch.cat([torch.max(torch.relu(c(x)), dim=2).values for c in self.convs], dim=1)
        x = self.drop(x)
        return self.fc(x).squeeze(-1) if self.regression else self.fc(x)


# 실제 학습 루프는 길어 별도 함수로
def train_textcnn(tr_t, tr_y, te_t, te_y, regression=False, epochs=10, batch=64, lr=1e-3):
    # tokenizer build, padding, train loop, eval
    # ... (구현 예정, GPU 환경에서)
    pass


In [ ]:
# Transformer (DistilBERT/BERT/RoBERTa) — head 교체로 분류/회귀 둘 다 처리
def train_transformer(model_id, tr_t, tr_y, te_t, te_y, regression=False,
                       max_len=256, epochs=3, batch=32, lr=2e-5):
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        Trainer, TrainingArguments,
    )
    n_label = 1 if regression else len(set(tr_y))
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=n_label,
        problem_type='regression' if regression else 'single_label_classification')
    # tokenize, Trainer, train, eval
    # ... (구현 예정)
    pass


## 2. 학습 스윕

02의 sweep 결과(polluted texts/labels)를 직접 메모리에서 사용하거나 ↓ 처럼 csv에서 재구성.


In [ ]:
# 학습 루프 — Colab GPU 환경에서 큐로 백그라운드 실행
# results = []
# for ds_name, (tr_texts, tr_y, te_texts, te_y) in datasets.items():
#     for pol_name, pol_cls in CLASSIFICATION_POLLUTERS.items():
#         for lvl in LEVEL_GRID:
#             pol = pol_cls(lvl, random_seed=42)
#             tr_p, tr_yp = pol.pollute(tr_texts, tr_y)
#             for model_name, train_fn in MODELS.items():
#                 metric = train_fn(tr_p, tr_yp, te_texts, te_y)
#                 results.append({...})
# pd.DataFrame(results).to_csv('results/text_train_metrics.csv', index=False)


---

다음: `04_scoreboard_text.ipynb` — r 분석, polluter hold-out, default vs tuned 가중치.
